**This script generates a dataset in which the downloaded comments are assessed for their topic and emotion**.

It was run on Kaggle. Because Kaggle has weekly and per-session GPU usage limits, the notebook was executed in multiple batches until all downloaded comments had been processed.

In [ ]:
import pandas as pd
# The route of the file of each audience has to be loaded
combined_df = pd.read_csv("route")
combined_df

**Evaluate the language of each comment**

In [ ]:
import torch
from transformers import pipeline

In [ ]:
import torch
from transformers import pipeline

# Check whether a GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create the pipeline on GPU if available
language_detector = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection",
    truncation=True  # Enable truncation
)


# Function to classify languages as "spanish", "english", or "other language"
def classify_language(text):
    if pd.isna(text) or text.strip() == "":
        return "unclassifiable"  # Handle empty values
    result = language_detector(text)[0]  # Get the model classification
    label = result["label"]

    return label
#    # Assign categories based on the result
#    if label == "en":
#        return "en"
#    if label == "es":
#        return "es"
#    else:
#        return "other language"

# Apply the function to the "comment" column
combined_df["language"] = combined_df["comment"].apply(classify_language)

In [ ]:
combined_df["language"].value_counts()

**Evaluate the sentiment of each comment**language_detector = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection",
    device=0 if torch.cuda.is_available() else -1,
    truncation=True  # Enable truncation
)

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1  # Use GPU if available

classifier = pipeline('sentiment-analysis', 
                      model="nlptown/bert-base-multilingual-uncased-sentiment",
                      device=device)


In [ ]:
#combined_df = combined_df[combined_df["language"] != "en"]
combined_df['comment'] = combined_df['comment'].astype(str)
comments = combined_df['comment'].tolist()
results = classifier(comments, truncation=True, batch_size=8)
results = pd.DataFrame(results)
results['label'] = results['label'].map({
    "1 star": "Negativo",
    "2 stars": "Negativo",
    "3 stars": "Neutral",
    "4 stars": "Positivo",
    "5 stars": "Positivo"
})

    
combined_df['emotion'] = results['label']


In [ ]:
print(combined_df['emotion'].value_counts())

In [ ]:
from transformers import pipeline
import torch

# Configure GPU use if available
device = 0 if torch.cuda.is_available() else -1


etiquetas = [
    "Comparison",
    "Information Exchange",
    "Advice request",
    "Advice Give",
    "First impression",
    "Opinion Request",
    "Opinion",
    "Insult",
    "Compliment",
    "Criticism",
    "Tribute",
    "Speculation",
    "Expression of personal feelings",
    "Greetings",
    "Thanking",
    "Joke",
    "Anecdote",
    "Plans",
    "Desires",
    "Event anticipation",
    "Unclassifiable",
    "Spam",
    "Medical treatment",
    "Pseudoscientific treatment",
    "Emotional Support"
]


# Initialize the pipeline with GPU
clasificador = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=device
)

In [ ]:
# Process comments in batches instead of one by one
#comentarios = combined_df['comment'].tolist()  # Convert the column to a list
#resultados = clasificador(comentarios, candidate_labels=etiquetas)  # Send batch to GPU

In [ ]:
import pandas as pd

comentarios = combined_df["comment"].fillna("").tolist()

print(f"Procesando {len(comentarios)} comentarios...\n")

resultados = clasificador(
    comentarios,
    candidate_labels=etiquetas,
    multi_label=True
)

filas = []

for i, (comentario, resultado) in enumerate(
    zip(comentarios, resultados),
    start=1
):
    print(f"Comentario {i}:")
    print(comentario)
    print("Etiquetas y pesos:")

    fila = {"comment": comentario}

    for etiqueta, peso in zip(resultado["labels"], resultado["scores"]):
        print(f"  - {etiqueta}: {peso:.4f}")
        fila[etiqueta] = peso

    print("-" * 50)
    filas.append(fila)

df_resultados = pd.DataFrame(filas)

print("Procesamiento completado.")
display(df_resultados)

In [ ]:
comentarios = combined_df['comment'].tolist()
total_comentarios = len(comentarios)
batch_size = 64
resultados = []

# Log frequency (every 5 batches, adjustable)
log_frequency = 5

print(f" Processing {total_comentarios} comments (batch_size={batch_size})...\n")

# Progress percentage with basic prints
for i in range(0, total_comentarios, batch_size):
    batch = comentarios[i:i + batch_size]
    resultados.extend(clasificador(batch, candidate_labels=etiquetas))
    
    # Progress logic
    procesados = min(i + batch_size, total_comentarios)
    if (i // batch_size) % log_frequency == 0 or procesados == total_comentarios:
        porcentaje = (procesados / total_comentarios) * 100
        print(f" Progress: {procesados}/{total_comentarios} comments ({porcentaje:.1f}%)")
        
        # Show memory usage if using GPU
        if device == 0:
            mem = torch.cuda.memory_reserved(device) / 1e9
            print(f"Memory ussage: {mem:.2f} GB")

In [ ]:
# 1. Extract ONLY the main label from each result
etiquetas_principales = [res['labels'][0] for res in resultados]

# 2. Create separate DataFrames
df_procesados = combined_df.iloc[:len(resultados)].copy()
df_procesados['etiqueta'] = etiquetas_principales  # Single column with the label

df_procesados

In [ ]:
#df_procesados.to_csv('carlos_stro_clasificado_newTaxo2.csv', index=False)

In [ ]:
df_procesados["etiqueta"].value_counts()